# NeuroGolf 5480.41 Public Score Under Current Scoring Rules

This notebook publishes a reproducible public-score artifact for **The 2026 NeuroGolf Championship**.

**Confirmed public score under the current scoring/parser rules: 5480.41.**

The attached dataset contains the exact submitted `submission.zip` with `400` ONNX files. The code cell verifies the content manifest and writes `submission.zip` for Kaggle code-submission scoring.

This is a controlled public refresh of the same vote-bearing notebook entry. It intentionally separates current-rule public scoring from older pre-reset/stale parser-sensitive scores.


## Method Sketch

Current-rule public artifact promoted by targeted public-family compression transfer and dataset mining: task128 Jazivxt, task037 Konbu May 7, task310 Soban, a 21-task Konbu May 7 batch, a 7-task Dimok/Imaad/Soban batch, a 4-task public tail, and task127 from the limprog dataset. Negative probes on task089/task017 and old localtree sparse/selector variants are intentionally excluded.

The practical rule is simple: publish a scored, parser-clean artifact with an exact manifest. Avoid broad unverified rewrites and avoid treating local cost as the final judge.


## Attribution

This release builds on the public NeuroGolf notebook ecosystem and my previous public artifacts. Useful public references include:

- `afr1ste/neurogolf-6285-95-public-score-open-solution`
- `artemnazemtsev/neurogolf-acking-multiple-tasks-part-3`
- `artemnazemtsev/neurogolf-logic-driven-ensembling-part-2`
- `artemnazemtsev/neurogolf-logic-driven-ensembling-part-4`
- `jonathanchan/ngc26-constraint-smart-logic-mix-blending`

The lineage is listed for traceability, not as a claim that every public branch transfers cleanly.


## Rebuild and Verify `submission.zip`

The dataset normally mounts as the original `submission.zip`. The fallback path also supports extracted `task*.onnx` files.

`EXPECTED_MANIFEST_SHA256` is the stable identity over all ONNX contents, independent of outer zip metadata.


In [ ]:
from pathlib import Path
import hashlib
import shutil
import zipfile

EXPECTED_FILE_COUNT = 400
EXPECTED_PUBLIC_SCORE = "5480.41"
EXPECTED_ZIP_SHA256 = "25f12cd90994fdaae60faa4f394b141a2ae4f4b6407103bb4bcfc1a5b0ea6fa7"
EXPECTED_MANIFEST_SHA256 = "e66d6e7a4a0fa7b3f14cf8e7c13a513e1852b3c6eaa36533d9076400e8d8dbe9"
OUTPUT_ZIP = Path("submission.zip")

input_root = Path("/kaggle/input")
preferred = input_root / "neurogolf-5480-41-controlled-public-artifact"
search_roots = [preferred] + sorted(input_root.glob("*5480-41*artifact*")) + sorted(input_root.glob("*neurogolf*artifact*"))


def manifest_from_zip(zip_path: Path):
    with zipfile.ZipFile(zip_path) as zf:
        names = sorted(name for name in zf.namelist() if name.endswith(".onnx"))
        lines = []
        for name in names:
            data = zf.read(name)
            lines.append(f"{name}\t{len(data)}\t{hashlib.sha256(data).hexdigest()}")
    manifest_hash = hashlib.sha256("\n".join(lines).encode()).hexdigest()
    return names, manifest_hash


def manifest_from_files(files):
    lines = []
    for path in sorted(files):
        data = path.read_bytes()
        lines.append(f"{path.name}\t{len(data)}\t{hashlib.sha256(data).hexdigest()}")
    manifest_hash = hashlib.sha256("\n".join(lines).encode()).hexdigest()
    return lines, manifest_hash


source_zip = None
for root in search_roots:
    if not root.exists():
        continue
    raw_archives = sorted(root.rglob("submission_zip.bin"))
    if raw_archives:
        source_zip = raw_archives[0]
        break
    candidates = sorted(root.rglob("submission.zip"))
    if candidates:
        source_zip = candidates[0]
        break

if source_zip is not None:
    shutil.copy2(source_zip, OUTPUT_ZIP)
    zip_sha = hashlib.sha256(OUTPUT_ZIP.read_bytes()).hexdigest()
    names, manifest_sha = manifest_from_zip(OUTPUT_ZIP)
    assert zip_sha == EXPECTED_ZIP_SHA256, (zip_sha, EXPECTED_ZIP_SHA256)
else:
    task_files = []
    for root in search_roots:
        if root.exists():
            task_files = sorted(root.rglob("task*.onnx"))
            if task_files:
                break
    if not task_files:
        raise FileNotFoundError("Could not find submission.zip or task*.onnx files under /kaggle/input")
    lines, manifest_sha = manifest_from_files(task_files)
    assert len(task_files) == EXPECTED_FILE_COUNT, len(task_files)
    with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
        for path in task_files:
            zf.write(path, arcname=path.name)
    names = [path.name for path in task_files]
    zip_sha = hashlib.sha256(OUTPUT_ZIP.read_bytes()).hexdigest()

assert len(names) == EXPECTED_FILE_COUNT, len(names)
assert manifest_sha == EXPECTED_MANIFEST_SHA256, (manifest_sha, EXPECTED_MANIFEST_SHA256)

print(f"Wrote {OUTPUT_ZIP} for NeuroGolf public score {EXPECTED_PUBLIC_SCORE}")
print(f"ONNX files: {len(names)}")
print(f"zip sha256: {zip_sha}")
print(f"manifest sha256: {manifest_sha}")


## Practical Notes

- This title and score refer to the current Kaggle scoring/parser rules, not older stale Code-page rows.
- Public score is the promotion gate; local scoring can be misleading on parser-sensitive graph rewrites.
- Publish exact artifacts, not just ideas. The manifest is what makes the notebook reproducible.
- This notebook keeps the same vote-bearing entry and updates it in place.
